[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/03-cnns-deep.ipynb)

# Convolutional Neural Networks
**Module 7 — Lesson 3 | Estimated time: 35 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU

## Learning Objectives
By the end of this notebook you will be able to:
- Understand `nn.Conv2d` parameters: kernel, stride, padding, dilation
- Calculate output dimensions and receptive fields
- Implement LeNet-5 from scratch
- Build a mini ResNet with residual blocks
- Apply batch normalisation and dropout for regularisation
- Train on MNIST and visualise learned feature maps

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. Conv2d Mechanics

The output spatial size of a conv layer is:
$$H_{out} = \left\lfloor\frac{H_{in} + 2p - d(k-1) - 1}{s} + 1\right\rfloor$$

where `p` = padding, `k` = kernel size, `d` = dilation, `s` = stride.

In [ ]:
def conv_output_size(h_in, kernel=3, stride=1, padding=0, dilation=1):
    return (h_in + 2*padding - dilation*(kernel-1) - 1) // stride + 1

print('Input 28x28 through different conv configs:')
configs = [
    dict(kernel=3, stride=1, padding=0),
    dict(kernel=3, stride=1, padding=1),  # same-padding
    dict(kernel=3, stride=2, padding=1),  # downsample
    dict(kernel=5, stride=1, padding=2),  # larger kernel
    dict(kernel=3, stride=1, padding=2, dilation=2),
]
for cfg in configs:
    out = conv_output_size(28, **cfg)
    print(f'  {cfg} -> {out}x{out}')

# Live demo
x = torch.randn(1, 1, 28, 28)  # (batch, channels, H, W)
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)
pool = nn.MaxPool2d(kernel_size=2, stride=2)
out_conv = conv(x)
out_pool = pool(out_conv)
print(f'\nInput:       {tuple(x.shape)}')
print(f'After conv:  {tuple(out_conv.shape)}')
print(f'After pool:  {tuple(out_pool.shape)}')

## 2. LeNet-5 from Scratch

LeNet-5 (LeCun, 1998) was the first practical CNN, designed for handwritten digit recognition.

```
Input (1×28×28)
  Conv1 (6 filters, 5×5) → Tanh → AvgPool
  Conv2 (16 filters, 5×5) → Tanh → AvgPool
  Flatten → FC(120) → FC(84) → FC(10)
```

In [ ]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Feature extractor
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, padding=2),  # (1,28,28)->(6,28,28)
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),      # (6,14,14)
            nn.Conv2d(6, 16, kernel_size=5),            # (16,10,10)
            nn.Tanh(),
            nn.AvgPool2d(kernel_size=2, stride=2),      # (16,5,5)
        )
        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

lenet = LeNet5()
test_in = torch.randn(2, 1, 28, 28)
test_out = lenet(test_in)
print('LeNet-5 output shape:', test_out.shape)
total = sum(p.numel() for p in lenet.parameters())
print(f'Parameters: {total:,}')
print(lenet)

## 3. Mini ResNet with Residual Blocks

Residual connections (`y = F(x) + x`) allow networks to be much deeper without vanishing gradients. If the dimensions differ, a 1×1 conv is used as a shortcut projection.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        # Shortcut (identity or projection)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)   # residual connection
        return F.relu(out)

class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.prep  = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU()
        )
        self.layer1 = ResidualBlock(16, 32, stride=2)
        self.layer2 = ResidualBlock(32, 64, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

resnet = MiniResNet()
out = resnet(torch.randn(2, 1, 28, 28))
print('MiniResNet output:', out.shape)
print(f'Parameters: {sum(p.numel() for p in resnet.parameters()):,}')

## 4. Train on MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_ds = torchvision.datasets.MNIST(root='/tmp/mnist', train=True,  download=True, transform=transform)
test_ds  = torchvision.datasets.MNIST(root='/tmp/mnist', train=False, download=True, transform=transform)

# Use a subset for speed
train_subset = torch.utils.data.Subset(train_ds, range(10000))
test_subset  = torch.utils.data.Subset(test_ds,  range(2000))

train_dl = DataLoader(train_subset, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_subset,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

model     = MiniResNet().to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_accs, test_accs = [], []
for epoch in range(5):
    model.train()
    correct = total = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimiser.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimiser.step()
        correct += (out.argmax(1) == yb).sum().item()
        total   += len(yb)
    train_accs.append(correct / total)

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total   += len(yb)
    test_accs.append(correct / total)
    print(f'Epoch {epoch+1}: train={train_accs[-1]:.3f}  test={test_accs[-1]:.3f}')

## 5. Visualising Feature Maps

Peek inside the network by extracting and plotting the output feature maps of the first conv layer.

In [ ]:
# Get one batch
model.eval()
imgs, labels = next(iter(test_dl))
img = imgs[0:1].to(device)  # single image

# Hook to capture feature maps
feature_maps = {}
def hook_fn(module, inp, out):
    feature_maps['prep_conv'] = out.detach().cpu()

hook = model.prep[0].register_forward_hook(hook_fn)
with torch.no_grad():
    _ = model(img)
hook.remove()

fmaps = feature_maps['prep_conv'][0]  # (16, 28, 28)
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    if i < fmaps.shape[0]:
        ax.imshow(fmaps[i].numpy(), cmap='viridis')
        ax.set_title(f'Filter {i+1}', fontsize=8)
    ax.axis('off')
plt.suptitle('Feature Maps after Prep Conv Layer', fontsize=12)
plt.tight_layout(); plt.show()

# Also plot the original image
plt.figure(figsize=(2.5, 2.5))
plt.imshow(imgs[0, 0].numpy(), cmap='gray')
plt.title(f'Original (label={labels[0].item()})')
plt.axis('off'); plt.show()

# Training accuracy curve
plt.figure(figsize=(7, 3))
plt.plot(range(1, 6), train_accs, marker='o', label='Train')
plt.plot(range(1, 6), test_accs,  marker='s', label='Test', linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.title('MiniResNet on MNIST (10k subset)')
plt.legend(); plt.tight_layout(); plt.show()

## Practice Exercises

**Exercise 1 — Batch Norm Modes**
Add `model.train()` and `model.eval()` deliberately in the wrong order during evaluation and observe how batch norm behaves differently (running stats vs batch stats). Compare accuracy with correct vs incorrect mode usage.

**Exercise 2 — Receptive Field Calculator**
Write a function that takes a list of (kernel, stride, dilation) tuples representing a CNN's conv layers and computes the effective receptive field at the output. Verify your LeNet-5 has the expected receptive field.

**Exercise 3 — Deeper ResNet**
Extend `MiniResNet` to include a third residual stage with 128 filters. Add `nn.Dropout2d(p=0.1)` after each residual block. Train on the full 60k MNIST training set for 10 epochs and report final test accuracy.